# Controlled comparison of labeling methods on noisy samples

This notebook mirrors the data-preparation and ground-truth construction of
`compare_soft_labeling_on_pure_samples.ipynb`, but evaluates the labeling
methods on **noisy samples** (samples that contain a mixture of two cell types)
instead of pure samples.

In [1]:
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple, Union
from time import time

import numpy as np
import pandas as pd
import json
import pysam
from tqdm import tqdm
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from utils import (
    filename2ctype,
    nested_defaultdict_to_dict,
    get_idx_from_sig,
    get_sig_from_idx,
    nrmse,
    compute_counts_sig_per_ctype_from_dataset,
    extract_complete_dataset,
    compute_num_reads_in_dataset,
    generate_bootstrap_sample_from_dataset,
    compute_pgt_sig_prior_from_counts,
    compute_nrmse_per_sig_length,
)
from methods import (
    naive_sl_without_pooling,
    generalized_naive_sl_without_pooling,
    sl_with_simple_archetypes,
)

%load_ext autoreload
%autoreload 2

## Data preparation

### Setting selection


We study two cell types and a single DMR under one of two settings:

- **`differentiated`**: one of the two selected cell types is the *target* cell
  type of the DMR (i.e. the DMR is differentially methylated for it), while the
  other is not. The two cell types are therefore distinguishable at this DMR.
- **`non_differentiated`**: *neither* of the two selected cell types is the
  target cell type of the DMR. The two cell types are not expected to be
  distinguishable at this DMR.

Change the single `SETTING` flag below to switch between the two settings for
the whole notebook.

In [2]:
# Single flag controlling the whole notebook.
SETTING = "non_differentiated"  # one of {"differentiated", "non_differentiated"}
assert SETTING in {"differentiated", "non_differentiated"}

N_CTYPES = 2  # we always select exactly two cell types

### Selection of the ctypes and DMRs



First we load the file containing the coverage per ctype per dmr.

In [3]:
df_coverage_per_ctype_per_dmr = pd.read_csv(Path("./data/region_coverage.csv"))
df_coverage_per_ctype_per_dmr.columns = [
    col if col != "Ovary+Endom-Ep" else "Ovary-Ep"
    for col in df_coverage_per_ctype_per_dmr.columns
]
atlas = pd.read_csv(Path("../../Data/Atlas.U25.l4.hg38.full.tsv"), sep="\t")
# add target column to df_coverage_per_ctype_per_dmr
map_region_key_to_target = atlas.set_index(["chr", "start", "end"])["target"].to_dict()
df_coverage_per_ctype_per_dmr["target"] = df_coverage_per_ctype_per_dmr.apply(
    lambda row: map_region_key_to_target.get((row["chr"], row["start"], row["end"])),
    axis=1,
)

labels_dict = json.load(open(Path("../../App/labels_dict.json"), "r", encoding="utf-8"))
labels_dict = {int(k): v for k, v in labels_dict.items()}
labels_list = [labels_dict[i] for i in range(len(labels_dict))]

We now select the two cell types and the DMR $d$ that maximize the minimum
coverage of the two selected cell types, under the constraints imposed by the
chosen `SETTING`:

- In the `differentiated` setting, cell type 0 is forced to be the target cell
  type of the DMR and cell type 1 is the non-target cell type with the highest
  coverage at that DMR.
- In the `non_differentiated` setting, both cell types are chosen among the
  non-target cell types with the highest coverage at that DMR.

In [4]:
best_dmr_idx = None
best_min_coverage = -1.0
best_ctypes = None

for dmr_idx, row in df_coverage_per_ctype_per_dmr.iterrows():
    if row["target"] not in labels_list:
        continue
    target_ctype_idx = labels_list.index(row["target"])
    coverage_per_ctype = row[labels_list].values.astype(float)

    if SETTING == "differentiated":
        # ctype 0 = target ctype, ctype 1 = best-covered non-target ctype
        coverage_excl_target = coverage_per_ctype.copy()
        coverage_excl_target[target_ctype_idx] = -1.0
        other_ctype_idx = int(np.argmax(coverage_excl_target))
        selected_idx = [target_ctype_idx, other_ctype_idx]
    else:  # non_differentiated
        # both ctypes = the two best-covered non-target ctypes
        coverage_excl_target = coverage_per_ctype.copy()
        coverage_excl_target[target_ctype_idx] = -1.0
        selected_idx = list(np.argsort(coverage_excl_target)[-N_CTYPES:])

    min_coverage = coverage_per_ctype[selected_idx].min()
    if min_coverage > best_min_coverage:
        best_min_coverage = min_coverage
        best_dmr_idx = dmr_idx
        best_ctypes = [labels_list[i] for i in selected_idx]

print(f"Setting: {SETTING}")
print(
    f"Best dmr: {best_dmr_idx}, Best ctypes: {best_ctypes}, "
    f"Best min coverage: {best_min_coverage}"
)
print(
    f"Target ctype of the dmr: {df_coverage_per_ctype_per_dmr.loc[best_dmr_idx, 'target']}"
)
print(
    f"Expected number of reads in the raw dataset: "
    f"{df_coverage_per_ctype_per_dmr.loc[best_dmr_idx, best_ctypes].sum()}"
)

Setting: non_differentiated
Best dmr: 950, Best ctypes: ['Endothel', 'Blood-T'], Best min coverage: 5887.0
Target ctype of the dmr: Gallbladder
Expected number of reads in the raw dataset: 12109


In [5]:
selected_ctypes = best_ctypes
selected_dmr_idx = best_dmr_idx
DMR_START_CPG = df_coverage_per_ctype_per_dmr.loc[selected_dmr_idx, "startCpG"]
DMR_FETCH_START = DMR_START_CPG - 50
DMR_END_CPG = df_coverage_per_ctype_per_dmr.loc[selected_dmr_idx, "endCpG"]
DMR_CHR = df_coverage_per_ctype_per_dmr.loc[selected_dmr_idx, "chr"]
print(
    df_coverage_per_ctype_per_dmr.loc[
        selected_dmr_idx,
        ["chr", "start", "end", "startCpG", "endCpG", "target"] + selected_ctypes,
    ]
)

chr               chr22
start          41460640
end            41461941
startCpG       27661520
endCpG         27661554
target      Gallbladder
Endothel           5887
Blood-T            6222
Name: 950, dtype: object


### Extraction of reads

Reads are organized in a nested dictionary. The first level of the dictionary is
the cell type, the second level is the number of CpG sites in the reads, and the
third level are the individual starting positions. The values are dictionaries
mapping the pattern of methylation to the number of reads with that pattern.

We first extract the raw reads from the files (so the pattern may contain unknown
values). We then split the reads with unknown values into smaller reads
(separated by the unknown values). Finally, we create an additional "complete"
dataset where we add to the reads of size n all the subsets of reads with
size > n (to be used position per position only).

In [6]:
# get the list of files paths per ctype
PATH_TO_DATA = Path(
    "/staging/leuven/stg_00118/methylDL/data/loyfer2023/hg38/data/GSE186458"
)
files_per_ctype = defaultdict(list)
for file in PATH_TO_DATA.glob("*.pat.gz"):
    ctype = filename2ctype(file.name)
    if ctype in selected_ctypes:
        files_per_ctype[ctype].append(file)
files_per_ctype = dict(files_per_ctype)

In [7]:
# Extract the raw reads dataset for the selected dmr and selected ctypes
raw_reads_dataset = defaultdict(
    lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
)
n_loops = sum(len(files) for files in files_per_ctype.values())
with tqdm(total=n_loops, desc="Extracting reads") as pbar:
    for ctype, files in files_per_ctype.items():
        for file in files:
            csi_file_path = str(file) + ".csi"
            tbx = pysam.TabixFile(str(file), index=csi_file_path)
            for row in tbx.fetch(DMR_CHR, DMR_FETCH_START, DMR_END_CPG + 1):
                parts = row.split("\t")
                read_start = int(parts[1])
                pattern = parts[2]
                n_reads = int(parts[3])
                read_end = read_start + len(pattern)  # exclusive
                # crop the read to the dmr region
                if read_end < DMR_START_CPG:
                    continue  # the read ends before the dmr start
                if read_start >= DMR_END_CPG:
                    continue  # the read starts after the dmr end
                cropped_pattern_start = max(read_start, DMR_START_CPG)
                cropped_pattern_end = min(read_end, DMR_END_CPG)
                cropped_pattern = pattern[
                    cropped_pattern_start
                    - read_start : cropped_pattern_end
                    - read_start
                ]
                if len(cropped_pattern) == 0:
                    continue  # the read does not overlap with the dmr
                # update the count of the cropped pattern in the raw reads dataset
                cropped_pattern = cropped_pattern.replace("C", "1").replace("T", "0")
                raw_reads_dataset[ctype][len(cropped_pattern)][cropped_pattern_start][
                    cropped_pattern
                ] += n_reads
                pbar.update(1)
n_reads_in_raw_dataset = compute_num_reads_in_dataset(raw_reads_dataset)
print(f"Number of reads in raw dataset: {n_reads_in_raw_dataset}")

Extracting reads: 5629it [00:02, 2400.92it/s]           

Number of reads in raw dataset: 12109


In [8]:
#
LIMIT_NUMBER_START_POSITIONS = 10 if SETTING == "non_differentiated" else None
if LIMIT_NUMBER_START_POSITIONS is not None:
    # first we get all the start positions extracted from the raw reads dataset
    all_start_positions = set()
    for ctype, length_dict in raw_reads_dataset.items():
        for length, start_dict in length_dict.items():
            all_start_positions.update(start_dict.keys())
    all_start_positions = sorted(list(all_start_positions))
    # then we limit the number of start positions to the first LIMIT_NUMBER_START_POSITIONS
    authorized_start_positions = all_start_positions[:LIMIT_NUMBER_START_POSITIONS]
    for ctype, length_dict in raw_reads_dataset.items():
        for length, start_dict in length_dict.items():
            for start in list(start_dict.keys()):
                if start not in authorized_start_positions:
                    del start_dict[start]

In [9]:
# Create the original read dataset by splitting the patterns at the unknown positions "."
og_dataset = defaultdict(
    lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
)
for ctype, length_dict in raw_reads_dataset.items():
    for read_length in sorted(length_dict.keys()):  # start from the shortest reads
        for start, pattern_counts in length_dict[read_length].items():
            for pattern, n_reads in pattern_counts.items():
                # split the pattern at the unknown positions "."
                subpatterns = pattern.split(".")
                subpattern_starts = []
                current_pos = start
                for subpattern in subpatterns:
                    subpattern_starts.append(current_pos)
                    current_pos += len(subpattern)
                # update the count of the subpatterns in the reads dataset
                for subpattern, subpattern_start in zip(subpatterns, subpattern_starts):
                    if len(subpattern) == 0:
                        continue  # skip empty subpatterns
                    og_dataset[ctype][len(subpattern)][subpattern_start][
                        subpattern
                    ] += n_reads
n_reads_in_og_dataset = compute_num_reads_in_dataset(og_dataset)
print(
    f"Number of reads in original dataset (reads splitted at unknown positions): "
    f"{n_reads_in_og_dataset}"
)

Number of reads in original dataset (reads splitted at unknown positions): 5463


In [10]:
# Create a complete dataset by splitting long patterns into all possible subpatterns
complete_og_dataset = extract_complete_dataset(og_dataset)
print(
    f"Number of reads in complete original dataset: "
    f"{compute_num_reads_in_dataset(complete_og_dataset)}"
)

Number of reads in complete original dataset: 50206


Now we create the dict of all pairs (pattern start, pattern length) (over all
ctypes), which represent the possible positions of signatures.

In [11]:
# first we identify the start_positions common to all ctypes
start_positions_per_ctype = defaultdict(set)
for ctype, length_dict in complete_og_dataset.items():
    for read_length, start_dict in length_dict.items():
        for start in start_dict.keys():
            start_positions_per_ctype[ctype].add(start)
common_start_positions = set.intersection(*start_positions_per_ctype.values())

# then we identify the (start, read_length) pairs common to all ctypes
signature_positions_per_ctype = defaultdict(lambda: defaultdict(set))
for ctype, length_dict in complete_og_dataset.items():
    for read_length, start_dict in length_dict.items():
        for start in start_dict.keys():
            if start not in common_start_positions:
                continue
            signature_positions_per_ctype[ctype][start].add(read_length)
signature_positions = defaultdict(set)
for start in common_start_positions:
    signature_positions[start] = set.intersection(
        *[signature_positions_per_ctype[ctype][start] for ctype in selected_ctypes]
    )
signature_positions = nested_defaultdict_to_dict(signature_positions)
del common_start_positions
del start_positions_per_ctype
del signature_positions_per_ctype
print("Signature positions (pattern start, pattern length) pairs common to all ctypes:")
signature_positions

Signature positions (pattern start, pattern length) pairs common to all ctypes:


{np.int64(27661520): {1, 2, 3, 4, 5, 6, 7, 8},
 27661521: {1, 2, 3, 4, 5, 6, 7},
 27661522: {1, 2, 3, 4, 5, 6, 7, 8, 9, 10},
 27661523: {1, 2, 3, 4, 5, 6, 7, 8, 9},
 27661524: {1, 2, 3, 4, 5, 6, 7, 8},
 27661525: {1, 2, 3, 4, 5, 6, 7},
 27661526: {1, 2, 3, 4, 5, 6},
 27661527: {1, 2, 3, 4, 5},
 27661528: {1, 2, 3, 4},
 27661529: {1, 2, 3, 4, 5, 6, 7},
 27661530: {1, 2, 3, 4, 5, 6},
 27661531: {1, 2, 3, 4, 5},
 27661532: {1, 2, 3, 4},
 27661533: {1, 2, 3},
 27661534: {1, 2},
 27661535: {1}}

## Ground truth $P(\mathrm{sig}\mid\mathrm{ctype})$ for the whole dataset



As in `compare_soft_labeling_on_pure_samples.ipynb`, we construct the pseudo
ground truth (PGT) for $P(\mathrm{signature}\mid\mathrm{ctype})$ over the whole
dataset by counting on the complete dataset (there is no pooling of counts
between signatures) and normalizing per cell type with the naive soft labels
without pooling.

In [12]:
# compute counts of signatures per ctype
complete_og_counts_sig_per_ctype = compute_counts_sig_per_ctype_from_dataset(
    complete_og_dataset, signature_positions, selected_ctypes, N_CTYPES
)  # with the complete dataset
og_counts_sig_per_ctype = compute_counts_sig_per_ctype_from_dataset(
    og_dataset, signature_positions, selected_ctypes, N_CTYPES
)  # with the reads dataset

# compute the PGT P(s)
complete_og_pgt_sig_prior = compute_pgt_sig_prior_from_counts(
    complete_og_counts_sig_per_ctype, signature_positions
)  # with the complete dataset
og_pgt_sig_prior = compute_pgt_sig_prior_from_counts(
    og_counts_sig_per_ctype, signature_positions
)  # with the reads dataset

# compute the ground truth P(sig | ctype) with naive soft labels without pooling
# (warning is expected here as signatures with zero counts in all ctypes lead to
# division by zero)
pgt_naive_sl_sig_given_ctype = naive_sl_without_pooling(
    complete_og_counts_sig_per_ctype
)

/vsc-hard-mounts/leuven-data/389/vsc38912/Projects/syto/EDA/extract_soft_labels_from_noisy_samples/utils.py:225: RuntimeWarning: invalid value encountered in divide
  counts_sig_per_ctype_.sum(axis=0) / counts_sig_per_ctype_.sum()


## Playground

First we want to observe the behaviors of the methods when we give them pure samples and the correct mixing proportions, but tell them to re-estimate the proportions.

In [ ]:
ctype_proba_per_sample, reestimated_ctype_proba_per_sample = (
    generalized_naive_sl_without_pooling(
        complete_og_counts_sig_per_ctype,
        np.array([[1.0, 0.0], [0.0, 1.0]]),
        reestimate_ctype_proba_per_sample=True,
        max_num_iterations=1000,
        delta_tol=1e-5,
        verbose=True,
    )
)
reestimated_ctype_proba_per_sample.round(4)

In [ ]:
{
    start: length_dict[4].round(2)
    for start, length_dict in ctype_proba_per_sample.items()
    if 4 in length_dict
}

In [ ]:
{
    start: length_dict[4].round(2)
    for start, length_dict in pgt_naive_sl_sig_given_ctype.items()
    if 4 in length_dict
}

Very impressive recovery capabilities of the correct mixing proportions.

## Construction of noisy samples

A **noisy sample** is a mixture of the two selected cell types: a fraction
$p_{b,c}$ of its reads is drawn from the pure read distribution of cell type $c$.

In [13]:
def build_ctype_prop_per_sample(
    base_noise_level: float,
    std_noise_on_noise: float,
    n_samples: int,
    n_ctypes: int = N_CTYPES,
    create_anchor_sample: bool = False,
) -> np.ndarray:
    """Build per-sample mixing proportions for a given noise level.

    Samples alternate the dominant cell type. The dominant cell type gets
    proportion ``1 - base_noise_level`` and the remaining ``base_noise_level`` is spread
    equally over the other cell types.
    """
    props = np.zeros((n_samples, n_ctypes))
    if create_anchor_sample:
        anchor_sample_prop = 0.9
        props[0, 0] = anchor_sample_prop
        props[0, 1:] = (1 - anchor_sample_prop) / (n_ctypes - 1)
    for b in range(1 if create_anchor_sample else 0, n_samples):
        dominant = b % n_ctypes
        props[b, dominant] = (
            1.0 - base_noise_level + np.random.normal(0, std_noise_on_noise)
        )
        others = [c for c in range(n_ctypes) if c != dominant]
        for c in others:
            props[b, c] = base_noise_level / len(others) + np.random.normal(
                0, std_noise_on_noise
            )
    props = np.divide(props, props.sum(axis=1, keepdims=True))  # normalize to sum to 1
    return props


def generate_noisy_samples_counts(
    source_dataset,
    selected_ctypes_,
    ctype_prop_per_sample: np.ndarray,
    sample_sizes: np.ndarray,
    sig_positions,
    n_ctypes: int,
):
    """Generate per-sample signature counts for noisy (mixed) samples.

    For each sample b and each cell type c, ``round(p_{b,c} * sample_size_b)``
    reads are drawn (with replacement) from the pure read distribution of c, and
    the resulting signature counts are summed across cell types (the cell type of
    origin is unknown at the sample level).

    Returns a dict ``{start: {read_length: np.ndarray of shape (n_samples, 2**read_length)}}``.
    """
    n_samples = ctype_prop_per_sample.shape[0]
    counts_sig_per_sample = {
        start: {
            read_length: np.zeros((n_samples, 2**read_length), dtype=int)
            for read_length in read_lengths
        }
        for start, read_lengths in sig_positions.items()
    }
    for b in range(n_samples):
        for c_idx, ctype in enumerate(selected_ctypes_):
            n_reads_c = int(round(ctype_prop_per_sample[b, c_idx] * sample_sizes[b]))
            if n_reads_c <= 0:
                continue
            single_ctype_dataset = {ctype: source_dataset[ctype]}
            sampled = generate_bootstrap_sample_from_dataset(
                single_ctype_dataset, dataset_size_to_sample=n_reads_c
            )
            sampled_counts = compute_counts_sig_per_ctype_from_dataset(
                sampled, sig_positions, [ctype], 1
            )
            for start, read_lengths in sampled_counts.items():
                for read_length, counts in read_lengths.items():
                    counts_sig_per_sample[start][read_length][b] += counts[0]
    return counts_sig_per_sample

## Experiment 1: deconvolution with oracle proportions

We construct noisy samples and deconvolve them with two methods, providing each
method with the **oracle** mixing proportions (the methods do not re-estimate
them):

- `generalized_naive_sl_without_pooling` (to be implemented later),
- `sl_with_simple_archetypes`.

We then plot the distribution of the NRMSE between the estimated
$P(\mathrm{sig}\mid\mathrm{ctype})$ and the ground-truth
$P(\mathrm{sig}\mid\mathrm{ctype})$ for each method, exactly as in
`compare_soft_labeling_on_pure_samples.ipynb`, except that the columns now vary a
measure of the **noise in the samples** (the contamination fraction) instead of
the dataset size.

In [18]:
N_SIMULATIONS = 20
N_NOISY_SAMPLES = 10  # number of noisy samples jointly deconvolved per simulation
SAMPLE_SIZE = 200  # number of reads per noisy sample
MAX_PATTERN_LENGTH_TO_CONSIDER_FOR_NRMSE = (
    5  # max pattern length to consider for NRMSE computation
)
MAX_NUM_ITERATIONS = 500  # max number of iterations for the EM algorithm
DELTA_TOL = 1e-5  # convergence threshold for the EM algorithm
noise_grid = np.array([0.0, 0.1, 0.25, 0.5])
std_noise_on_noise = 0.1  # standard deviation of the noise added to the noise level
min_count_threshold_per_ctype_for_nrmse = 5

methods_exp1 = ["sl_with_simple_archetypes"] # "generalized_naive_sl_without_pooling", 

In [20]:
# Reference counts used to decide which (sig, ctype) pairs are considered in the
# NRMSE (those with ground-truth support). Kept fixed across simulations.
ref_counts_for_nrmse = og_counts_sig_per_ctype

# Store raw predictions per noise level and method
results_exp1_predictions = {
    noise_level: {method: [] for method in methods_exp1} for noise_level in noise_grid
}
# store the execution time ratios (signature/archetype) per noise level
time_ratios_exp1_per_noise_level = {noise_level: [] for noise_level in noise_grid}

n_loops = len(noise_grid) * N_SIMULATIONS
with tqdm(total=n_loops, desc="Exp 1: oracle proportions") as pbar:
    for noise_level in noise_grid:
        for sim_id in range(N_SIMULATIONS):
            ctype_prop_per_sample = build_ctype_prop_per_sample(
                noise_level,
                std_noise_on_noise,
                N_NOISY_SAMPLES,
                N_CTYPES,
                create_anchor_sample=False,
            )
            sample_sizes = np.full(N_NOISY_SAMPLES, SAMPLE_SIZE, dtype=int)
            counts_sig_per_sample = generate_noisy_samples_counts(
                og_dataset,
                selected_ctypes,
                ctype_prop_per_sample,
                sample_sizes,
                signature_positions,
                N_CTYPES,
            )

            # both methods receive the oracle proportions and do not re-estimate them
            if "generalized_naive_sl_without_pooling" in methods_exp1:
                start_sig_time = time()
                generalized_sig_given_ctype = generalized_naive_sl_without_pooling(
                    counts_sig_per_sample,
                    ctype_prop_per_sample,
                    reestimate_ctype_proba_per_sample=False,
                    max_num_iterations=MAX_NUM_ITERATIONS,
                    delta_tol=DELTA_TOL,
                )
                end_sig_time = time()
            if "sl_with_simple_archetypes" in methods_exp1:
                start_archetype_time = time()
                simple_archetypes_sig_given_ctype = sl_with_simple_archetypes(
                    counts_sig_per_sample,
                    ctype_prop_per_sample,
                    reestimate_ctype_proba_per_sample=False,
                    max_num_iterations=MAX_NUM_ITERATIONS,
                    delta_tol=DELTA_TOL,
                )
                end_archetype_time = time()

            # Store the execution time ratio (signature/archetype)
            if "generalized_naive_sl_without_pooling" in methods_exp1 and "sl_with_simple_archetypes" in methods_exp1:
                time_ratio = (end_sig_time - start_sig_time) / (
                    end_archetype_time - start_archetype_time
                )
                time_ratios_exp1_per_noise_level[noise_level].append(time_ratio)

            # Store raw predictions
            if "generalized_naive_sl_without_pooling" in methods_exp1:
                results_exp1_predictions[noise_level][
                    "generalized_naive_sl_without_pooling"
                ].append(generalized_sig_given_ctype)
            if "sl_with_simple_archetypes" in methods_exp1:
                results_exp1_predictions[noise_level]["sl_with_simple_archetypes"].append(
                    simple_archetypes_sig_given_ctype
                )
            pbar.update(1)

Exp 1: oracle proportions: 100%|██████████| 80/80 [02:28<00:00,  1.86s/it]


In [21]:
# Compute NRMSE from the stored predictions
results_exp1_per_noise_per_read_length_per_method = {
    noise_level: {
        read_length: {method: [] for method in methods_exp1}
        for read_length in range(1, MAX_PATTERN_LENGTH_TO_CONSIDER_FOR_NRMSE + 1)
    }
    for noise_level in noise_grid
}

for noise_level in noise_grid:
    for method_name in methods_exp1:
        for predicted_sig_given_ctype in results_exp1_predictions[noise_level][
            method_name
        ]:
            nrmse_per_length, _ = compute_nrmse_per_sig_length(
                predicted_sig_given_ctype,
                pgt_naive_sl_sig_given_ctype,
                counts_sig_per_ctype=ref_counts_for_nrmse,
                min_count_threshold_per_ctype=min_count_threshold_per_ctype_for_nrmse,
            )
            for read_length in range(1, MAX_PATTERN_LENGTH_TO_CONSIDER_FOR_NRMSE + 1):
                if read_length in nrmse_per_length:
                    results_exp1_per_noise_per_read_length_per_method[noise_level][
                        read_length
                    ][method_name].append(nrmse_per_length[read_length])

In [ ]:
method_labels_exp1 = ["Simple\narchetypes"]#["Generalized naive\n(no pooling)", "Simple\narchetypes"]
colors_exp1 = ["tab:red"]#["tab:blue", "tab:red"]

fig, axes = plt.subplots(
    MAX_PATTERN_LENGTH_TO_CONSIDER_FOR_NRMSE,
    len(noise_grid),
    figsize=(4 * len(noise_grid), 4 * MAX_PATTERN_LENGTH_TO_CONSIDER_FOR_NRMSE),
    sharey="row",
)
USE_VIOLIN_PLOTS = False
for row_idx, read_length in enumerate(
    range(1, MAX_PATTERN_LENGTH_TO_CONSIDER_FOR_NRMSE + 1)
):
    for col_idx, noise_level in enumerate(noise_grid):
        ax = axes[row_idx, col_idx]
        data_to_plot = [
            results_exp1_per_noise_per_read_length_per_method[noise_level][read_length][
                method
            ]
            for method in methods_exp1
        ]
        if USE_VIOLIN_PLOTS:
            vp = ax.violinplot(
                data_to_plot, showmeans=False, showmedians=True  # , bw_method="scott"
            )
            for patch, color in zip(vp["bodies"], colors_exp1):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
            # for median in vp["cmedians"]:
            #     median.set_color("black")
        else:
            bp = ax.boxplot(data_to_plot, patch_artist=True)
            for patch, color in zip(bp["boxes"], colors_exp1):
                patch.set_facecolor(color)
            for median in bp["medians"]:
                median.set_color("black")
        ax.set_ylim(0, 1)
        ax.grid(axis="y", linestyle="--", alpha=0.7)
        ax.set_xticks([])
        if row_idx == 0:
            ax.set_title(f"Base contamination= {noise_level:.2f}")
        ax.set_ylabel(f"Read length={read_length}\nNRMSE" if col_idx == 0 else "")

legend_handles = [
    Patch(facecolor=color, edgecolor="black", label=label)
    for color, label in zip(colors_exp1, method_labels_exp1)
]
fig.legend(
    handles=legend_handles,
    title="Methods",
    loc="center left",
    bbox_to_anchor=(0.85, 0.5),
    frameon=True,
    fancybox=True,
    framealpha=1.0,
    borderpad=0.8,
)
fig.suptitle(
    "Using oracle ctype proportions: NRMSE of estimated P(signature | ctype)"
    f"vs sample noise level, per pattern length and method"
    f"\nSetting: {SETTING}, {N_CTYPES} cell types ({selected_ctypes}), "
    f"#noisy samples: {N_NOISY_SAMPLES}, sample size: {SAMPLE_SIZE}, "
    f"#simulations: {N_SIMULATIONS}, std_noise_on_noise: {std_noise_on_noise}"
    f"\nmin counts per ctype to be included in NRMSE: {min_count_threshold_per_ctype_for_nrmse}, max iterations: {MAX_NUM_ITERATIONS}, delta tol: {DELTA_TOL}",
    y=1.0,
)
plt.tight_layout(rect=[0, 0, 0.86, 1])
plt.savefig(
    Path(
        f"./output/noisy_samples_oracle_proportions_nrmse_setting={SETTING}_noisy-samples={N_NOISY_SAMPLES}_sample-size={SAMPLE_SIZE}_max-iterations={MAX_NUM_ITERATIONS}_delta-tol={DELTA_TOL}.pdf"
    ),
    bbox_inches="tight",
)
plt.show()

In [ ]:
# plot the boxplots of the execution time ratios (signature/archetype) per noise level
# everything on a single plot, with the noise level on the x-axis and the execution time ratio on the y-axis
fig, ax = plt.subplots(figsize=(8, 6))
data_to_plot = [
    time_ratios_exp1_per_noise_level[noise_level] for noise_level in noise_grid
]
bp = ax.boxplot(data_to_plot, patch_artist=True)
for patch, color in zip(bp["boxes"], ["tab:blue"] * len(noise_grid)):
    patch.set_facecolor(color)
for median in bp["medians"]:
    median.set_color("black")
ax.set_xticks(range(1, len(noise_grid) + 1))
ax.set_xticklabels([f"{noise_level:.2f}" for noise_level in noise_grid])
ax.set_xlabel("Base contamination level")
ax.set_ylabel("Execution time ratio (signature/archetype)")
ax.set_title(
    "Execution time ratio (signature/archetype) vs base contamination level"
    f"\nSetting: {SETTING}, {N_CTYPES} cell types ({selected_ctypes}), "
    f"#noisy samples: {N_NOISY_SAMPLES}, sample size: {SAMPLE_SIZE}, "
    f"#simulations: {N_SIMULATIONS}, std_noise_on_noise: {std_noise_on_noise}"
    f"\nmax iterations: {MAX_NUM_ITERATIONS}, delta tol: {DELTA_TOL}"
)
ax.grid(axis="y", linestyle="--", alpha=0.7)
ax.set_yticks(np.linspace(0, 7.5, 16))
ax.set_ylim(0, 7.5)
plt.tight_layout()
plt.savefig(
    Path(
        f"./output/noisy_samples_oracle_proportions_time_ratio_setting={SETTING}_noisy-samples={N_NOISY_SAMPLES}_sample-size={SAMPLE_SIZE}_max-iterations={MAX_NUM_ITERATIONS}_delta-tol={DELTA_TOL}.pdf"
    ),
    bbox_inches="tight",
)
plt.show()

## Experiment 2: deconvolution with a noisy estimate of the proportions

Same setup as Experiment 1, but the methods now receive a **noisy estimate** of
the mixing proportions instead of the oracle values, and are allowed to
re-estimate them (`reestimate_ctype_proba_per_sample=True`). We add Gaussian
noise to the true proportions and renormalize to obtain the initial estimate
passed to each method.

We plot two quantities:

- the NRMSE between the estimated $P(\mathrm{sig}\mid\mathrm{ctype})$ and the
  ground truth (as in Experiment 1), and
- the NRMSE between the proportions estimated by the methods and the true mixing
  proportions of the samples.

In [ ]:
PROP_ESTIMATE_NOISE_STD = 0.1  # std of the Gaussian noise added to the true proportions
N_SIMULATIONS = 100
N_NOISY_SAMPLES = 10
MAX_NUM_ITERATIONS = 100
DELTA_TOL = 1e-4
SAMPLE_SIZE = 200
std_noise_on_noise = 0.1  # standard deviation of the noise added to the noise level
min_count_threshold_per_ctype_for_nrmse = 5
noise_grid = np.array([0.0, 0.1, 0.25, 0.5])

In [ ]:
methods_exp2 = ["generalized_naive_sl_without_pooling", "sl_with_simple_archetypes"]

rng = np.random.default_rng(0)

# Store raw predictions and raw estimated proportions per simulation
results_exp2_predictions = {
    noise_level: {method: [] for method in methods_exp2} for noise_level in noise_grid
}
results_exp2_estimated_props = {
    noise_level: {method: [] for method in methods_exp2} for noise_level in noise_grid
}
results_exp2_true_props = {noise_level: [] for noise_level in noise_grid}
# Store execution time ratios (generalized/archetype) per noise level
time_ratios_exp2_per_noise_level = {noise_level: [] for noise_level in noise_grid}

n_loops = len(noise_grid) * N_SIMULATIONS
with tqdm(total=n_loops, desc="Exp 2: noisy proportion estimate") as pbar:
    for noise_level in noise_grid:
        for sim_id in range(N_SIMULATIONS):
            ctype_prop_per_sample = build_ctype_prop_per_sample(
                noise_level, std_noise_on_noise, N_NOISY_SAMPLES, N_CTYPES
            )
            sample_sizes = np.full(N_NOISY_SAMPLES, SAMPLE_SIZE, dtype=int)
            counts_sig_per_sample = generate_noisy_samples_counts(
                og_dataset,
                selected_ctypes,
                ctype_prop_per_sample,
                sample_sizes,
                signature_positions,
                N_CTYPES,
            )

            # noisy estimate of the proportions given to the methods
            noisy_prop_estimate = perturb_proportions(
                ctype_prop_per_sample, PROP_ESTIMATE_NOISE_STD, rng
            )

            # Store ground-truth proportions for this simulation
            results_exp2_true_props[noise_level].append(ctype_prop_per_sample.copy())

            elapsed_generalized = None
            elapsed_archetype = None
            for method_name in methods_exp2:
                start_method_time = time()
                if method_name == "generalized_naive_sl_without_pooling":
                    estimated_sig_given_ctype, estimated_props = (
                        generalized_naive_sl_without_pooling(
                            counts_sig_per_sample,
                            noisy_prop_estimate,
                            reestimate_ctype_proba_per_sample=True,
                            max_num_iterations=MAX_NUM_ITERATIONS,
                            delta_tol=DELTA_TOL,
                        )
                    )
                elif method_name == "sl_with_simple_archetypes":
                    estimated_sig_given_ctype, estimated_props = (
                        sl_with_simple_archetypes(
                            counts_sig_per_sample,
                            noisy_prop_estimate,
                            reestimate_ctype_proba_per_sample=True,
                            max_num_iterations=MAX_NUM_ITERATIONS,
                            delta_tol=DELTA_TOL,
                        )
                    )
                else:
                    raise ValueError(f"Unknown method: {method_name}")
                end_method_time = time()

                # Track runtime per method for this simulation
                elapsed_time = end_method_time - start_method_time
                if method_name == "generalized_naive_sl_without_pooling":
                    elapsed_generalized = elapsed_time
                elif method_name == "sl_with_simple_archetypes":
                    elapsed_archetype = elapsed_time

                # Store raw outputs
                results_exp2_predictions[noise_level][method_name].append(
                    estimated_sig_given_ctype
                )
                results_exp2_estimated_props[noise_level][method_name].append(
                    estimated_props
                )

            # Store execution time ratio (generalized/archetype)
            if elapsed_generalized is not None and elapsed_archetype not in (None, 0):
                time_ratios_exp2_per_noise_level[noise_level].append(
                    elapsed_generalized / elapsed_archetype
                )
            pbar.update(1)

In [ ]:
# Compute NRMSE metrics from stored raw outputs
results_exp2_sig_given_ctype = {
    noise_level: {
        read_length: {method: [] for method in methods_exp2}
        for read_length in range(1, 6)
    }
    for noise_level in noise_grid
}
results_exp2_proportions = {
    noise_level: {method: [] for method in methods_exp2} for noise_level in noise_grid
}

ref_counts_for_nrmse = og_counts_sig_per_ctype
for noise_level in noise_grid:
    for method_name in methods_exp2:
        for sim_id, estimated_sig_given_ctype in enumerate(
            results_exp2_predictions[noise_level][method_name]
        ):
            nrmse_per_length, _ = compute_nrmse_per_sig_length(
                estimated_sig_given_ctype,
                pgt_naive_sl_sig_given_ctype,
                counts_sig_per_ctype=ref_counts_for_nrmse,
                min_count_threshold_per_ctype=min_count_threshold_per_ctype_for_nrmse,
            )
            for read_length in range(1, 6):
                if read_length in nrmse_per_length:
                    results_exp2_sig_given_ctype[noise_level][read_length][
                        method_name
                    ].append(nrmse_per_length[read_length])

            estimated_props = results_exp2_estimated_props[noise_level][method_name][
                sim_id
            ]
            true_props = results_exp2_true_props[noise_level][sim_id]
            proportion_nrmse = nrmse(true_props, estimated_props)
            results_exp2_proportions[noise_level][method_name].append(proportion_nrmse)

In [ ]:
# Plot the NRMSE of P(sig | ctype) for Experiment 2
method_labels_exp2 = ["Generalized naive\n(no pooling)", "Simple\narchetypes"]
colors_exp2 = ["tab:blue", "tab:red"]

fig, axes = plt.subplots(
    5, len(noise_grid), figsize=(4 * len(noise_grid), 4 * 5), sharey="row"
)

for row_idx, read_length in enumerate(range(1, 6)):
    for col_idx, noise_level in enumerate(noise_grid):
        ax = axes[row_idx, col_idx]
        data_to_plot = [
            results_exp2_sig_given_ctype[noise_level][read_length][method]
            for method in methods_exp2
        ]
        bp = ax.boxplot(data_to_plot, patch_artist=True)
        for patch, color in zip(bp["boxes"], colors_exp2):
            patch.set_facecolor(color)
        for median in bp["medians"]:
            median.set_color("black")
        ax.set_ylim(0, 1.0)
        ax.grid(axis="y", linestyle="--", alpha=0.7)
        ax.set_xticks([])
        if row_idx == 0:
            ax.set_title(f"Noise level (contamination)\n= {noise_level:.2f}")
        ax.set_ylabel(f"Read length={read_length}\nNRMSE" if col_idx == 0 else "")

legend_handles = [
    Patch(facecolor=color, edgecolor="black", label=label)
    for color, label in zip(colors_exp2, method_labels_exp2)
]
fig.legend(
    handles=legend_handles,
    title="Methods",
    loc="center left",
    bbox_to_anchor=(0.85, 0.5),
    frameon=True,
    fancybox=True,
    framealpha=1.0,
    borderpad=0.8,
)
fig.suptitle(
    f"Using proportion estimates): NRMSE of estimated P(signature | ctype) vs "
    f"sample noise level, per pattern length and method"
    f"\nSetting: {SETTING}, {N_CTYPES} cell types ({selected_ctypes}), "
    f"prop. estimate noise std: {PROP_ESTIMATE_NOISE_STD}, "
    f"#simulations: {N_SIMULATIONS}"
    f"max iterations: {MAX_NUM_ITERATIONS}, delta tol: {DELTA_TOL}",
    y=1.0,
)
plt.tight_layout(rect=[0, 0, 0.86, 1])
plt.savefig(
    Path(
        f"./output/noisy_samples_noisy_proportions_nrmse_setting={SETTING}_noisy-samples={N_NOISY_SAMPLES}_sample-size={SAMPLE_SIZE}_max-iterations={MAX_NUM_ITERATIONS}_delta-tol={DELTA_TOL}_prop-estimate-noise-std={PROP_ESTIMATE_NOISE_STD}.pdf"
    ),
    bbox_inches="tight",
)
plt.show()

In [ ]:
# Plot the NRMSE of the estimated proportions for Experiment 2
fig, axes = plt.subplots(
    1, len(noise_grid), figsize=(4 * len(noise_grid), 4), sharey=True
)
if len(noise_grid) == 1:
    axes = [axes]

for col_idx, noise_level in enumerate(noise_grid):
    ax = axes[col_idx]
    data_to_plot = [
        results_exp2_proportions[noise_level][method] for method in methods_exp2
    ]
    bp = ax.boxplot(data_to_plot, patch_artist=True)
    for patch, color in zip(bp["boxes"], colors_exp2):
        patch.set_facecolor(color)
    for median in bp["medians"]:
        median.set_color("black")
    ax.set_ylim(0, 1.0)
    ax.grid(axis="y", linestyle="--", alpha=0.7)
    ax.set_xticks([])
    ax.set_title(f"Noise level (contamination)\n= {noise_level:.2f}")
    if col_idx == 0:
        ax.set_ylabel("NRMSE of estimated proportions")

legend_handles = [
    Patch(facecolor=color, edgecolor="black", label=label)
    for color, label in zip(colors_exp2, method_labels_exp2)
]
fig.legend(
    handles=legend_handles,
    title="Methods",
    loc="center left",
    bbox_to_anchor=(0.9, 0.5),
    frameon=True,
    fancybox=True,
    framealpha=1.0,
    borderpad=0.8,
)
fig.suptitle(
    "Exp 2 (noisy proportion estimate): NRMSE of the estimated ctype proportions vs "
    "sample noise level, per method"
    f"\nSetting: {SETTING}, prop. estimate noise std: {PROP_ESTIMATE_NOISE_STD}, "
    f"#simulations: {N_SIMULATIONS}"
    f"\nmax iterations: {MAX_NUM_ITERATIONS}, delta tol: {DELTA_TOL}",
    y=1.05,
)
plt.savefig(
    Path(
        f"./output/noisy_samples_noisy_proportions_ctype_prop_nrmse_setting={SETTING}_noisy-samples={N_NOISY_SAMPLES}_sample-size={SAMPLE_SIZE}_max-iterations={MAX_NUM_ITERATIONS}_delta-tol={DELTA_TOL}_prop-estimate-noise-std={PROP_ESTIMATE_NOISE_STD}.pdf"
    ),
    bbox_inches="tight",
)
plt.tight_layout(rect=[0, 0, 0.88, 1])
plt.show()

In [ ]:
# Plot execution time ratios (generalized/archetype) for Experiment 2
fig, ax = plt.subplots(figsize=(8, 6))
data_to_plot = [
    time_ratios_exp2_per_noise_level[noise_level] for noise_level in noise_grid
]
bp = ax.boxplot(data_to_plot, patch_artist=True)
for patch, color in zip(bp["boxes"], ["tab:blue"] * len(noise_grid)):
    patch.set_facecolor(color)
for median in bp["medians"]:
    median.set_color("black")
ax.set_xticks(range(1, len(noise_grid) + 1))
ax.set_xticklabels([f"{noise_level:.2f}" for noise_level in noise_grid])
ax.set_xlabel("Base contamination level")
ax.set_ylabel("Execution time ratio (generalized/archetype)")
ax.set_title(
    "Exp 2: execution time ratio (generalized/archetype) vs base contamination level"
    f"\nSetting: {SETTING}, {N_CTYPES} cell types ({selected_ctypes}), "
    f"#noisy samples: {N_NOISY_SAMPLES}, sample size: {SAMPLE_SIZE}, "
    f"\n#simulations: {N_SIMULATIONS}, prop. estimate noise std: {PROP_ESTIMATE_NOISE_STD},"
    f"max iterations: {MAX_NUM_ITERATIONS}, delta tol: {DELTA_TOL}"
)
ax.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.savefig(
    Path(
        f"./output/noisy_samples_noisy_proportions_time_ratio_setting={SETTING}_"
        f"noisy-samples={N_NOISY_SAMPLES}_sample-size={SAMPLE_SIZE}_"
        f"max-iterations={MAX_NUM_ITERATIONS}_delta-tol={DELTA_TOL}.pdf"
    ),
    bbox_inches="tight",
)
plt.show()

## Experiment 3: deconvolution with varying noise on the proportion estimate

Same setup as Experiment 2, but we now **fix** the contamination noise of the
cell-type proportions in the samples (`FIXED_NOISE_LEVEL`) and instead **vary**
the standard deviation of the Gaussian noise added to the proportion estimate
passed to the methods (`prop_estimate_noise_grid`). Each method still
re-estimates the proportions (`reestimate_ctype_proba_per_sample=True`).

We plot two quantities, now as a function of the **proportion estimate noise
std** (on the x axis / columns):

- the NRMSE between the estimated $P(\mathrm{sig}\mid\mathrm{ctype})$ and the
  ground truth, and
- the NRMSE between the proportions estimated by the methods and the true mixing
  proportions of the samples.

In [ ]:
def exp3_perturb_proportions(
    true_props: np.ndarray, noise_std: float, rng
) -> np.ndarray:
    """Return a noisy estimate of the proportions (clipped and renormalized)."""
    noisy = true_props + rng.normal(0.0, noise_std, size=true_props.shape)
    noisy = np.clip(noisy, 1e-3, None)
    noisy = noisy / noisy.sum(axis=1, keepdims=True)
    return noisy

In [ ]:
# Experiment 3 configuration: fix the sample contamination noise, vary the
# proportion estimate noise std. All parameters needed to run Experiment 3
# independently from Experiment 2 are defined here.
FIXED_NOISE_LEVEL = (
    0.3  # fixed contamination noise of the ctype proportions in the samples
)
N_SIMULATIONS = 30
N_NOISY_SAMPLES = 10
MAX_NUM_ITERATIONS = 100
DELTA_TOL = 1e-4
SAMPLE_SIZE = 200
std_noise_on_noise = 0.1  # standard deviation of the noise added to the noise level
min_count_threshold_per_ctype_for_nrmse = 5
# grid of std values for the Gaussian noise added to the true proportions
prop_estimate_noise_grid = np.array([0.0, 0.1, 0.25, 0.5])

methods_exp3 = ["generalized_naive_sl_without_pooling", "sl_with_simple_archetypes"]
method_labels_exp3 = ["Generalized naive\n(no pooling)", "Simple\narchetypes"]
colors_exp3 = ["tab:blue", "tab:red"]

In [ ]:
rng = np.random.default_rng(0)

# Store raw predictions and raw estimated proportions per simulation
results_exp3_predictions = {
    prop_noise: {method: [] for method in methods_exp3}
    for prop_noise in prop_estimate_noise_grid
}
results_exp3_estimated_props = {
    prop_noise: {method: [] for method in methods_exp3}
    for prop_noise in prop_estimate_noise_grid
}
results_exp3_true_props = {prop_noise: [] for prop_noise in prop_estimate_noise_grid}
# Store execution time ratios (generalized/archetype) per proportion estimate noise std
time_ratios_exp3_per_prop_noise = {
    prop_noise: [] for prop_noise in prop_estimate_noise_grid
}

n_loops = len(prop_estimate_noise_grid) * N_SIMULATIONS
with tqdm(total=n_loops, desc="Exp 3: varying proportion estimate noise") as pbar:
    for prop_noise in prop_estimate_noise_grid:
        for sim_id in range(N_SIMULATIONS):
            ctype_prop_per_sample = build_ctype_prop_per_sample(
                FIXED_NOISE_LEVEL, std_noise_on_noise, N_NOISY_SAMPLES, N_CTYPES
            )
            sample_sizes = np.full(N_NOISY_SAMPLES, SAMPLE_SIZE, dtype=int)
            counts_sig_per_sample = generate_noisy_samples_counts(
                og_dataset,
                selected_ctypes,
                ctype_prop_per_sample,
                sample_sizes,
                signature_positions,
                N_CTYPES,
            )

            # noisy estimate of the proportions given to the methods
            noisy_prop_estimate = exp3_perturb_proportions(
                ctype_prop_per_sample, prop_noise, rng
            )

            # Store ground-truth proportions for this simulation
            results_exp3_true_props[prop_noise].append(ctype_prop_per_sample.copy())

            elapsed_generalized = None
            elapsed_archetype = None
            for method_name in methods_exp3:
                start_method_time = time()
                if method_name == "generalized_naive_sl_without_pooling":
                    estimated_sig_given_ctype, estimated_props = (
                        generalized_naive_sl_without_pooling(
                            counts_sig_per_sample,
                            noisy_prop_estimate,
                            reestimate_ctype_proba_per_sample=True,
                            max_num_iterations=MAX_NUM_ITERATIONS,
                            delta_tol=DELTA_TOL,
                        )
                    )
                elif method_name == "sl_with_simple_archetypes":
                    estimated_sig_given_ctype, estimated_props = (
                        sl_with_simple_archetypes(
                            counts_sig_per_sample,
                            noisy_prop_estimate,
                            reestimate_ctype_proba_per_sample=True,
                            max_num_iterations=MAX_NUM_ITERATIONS,
                            delta_tol=DELTA_TOL,
                        )
                    )
                else:
                    raise ValueError(f"Unknown method: {method_name}")
                end_method_time = time()

                # Track runtime per method for this simulation
                elapsed_time = end_method_time - start_method_time
                if method_name == "generalized_naive_sl_without_pooling":
                    elapsed_generalized = elapsed_time
                elif method_name == "sl_with_simple_archetypes":
                    elapsed_archetype = elapsed_time

                # Store raw outputs
                results_exp3_predictions[prop_noise][method_name].append(
                    estimated_sig_given_ctype
                )
                results_exp3_estimated_props[prop_noise][method_name].append(
                    estimated_props
                )

            # Store execution time ratio (generalized/archetype)
            if elapsed_generalized is not None and elapsed_archetype not in (None, 0):
                time_ratios_exp3_per_prop_noise[prop_noise].append(
                    elapsed_generalized / elapsed_archetype
                )
            pbar.update(1)

In [ ]:
# Compute NRMSE metrics from stored raw outputs (Experiment 3)
results_exp3_sig_given_ctype = {
    prop_noise: {
        read_length: {method: [] for method in methods_exp3}
        for read_length in range(1, 6)
    }
    for prop_noise in prop_estimate_noise_grid
}
results_exp3_proportions = {
    prop_noise: {method: [] for method in methods_exp3}
    for prop_noise in prop_estimate_noise_grid
}

ref_counts_for_nrmse = og_counts_sig_per_ctype
for prop_noise in prop_estimate_noise_grid:
    for method_name in methods_exp3:
        for sim_id, estimated_sig_given_ctype in enumerate(
            results_exp3_predictions[prop_noise][method_name]
        ):
            nrmse_per_length, _ = compute_nrmse_per_sig_length(
                estimated_sig_given_ctype,
                pgt_naive_sl_sig_given_ctype,
                counts_sig_per_ctype=ref_counts_for_nrmse,
                min_count_threshold_per_ctype=min_count_threshold_per_ctype_for_nrmse,
            )
            for read_length in range(1, 6):
                if read_length in nrmse_per_length:
                    results_exp3_sig_given_ctype[prop_noise][read_length][
                        method_name
                    ].append(nrmse_per_length[read_length])

            estimated_props = results_exp3_estimated_props[prop_noise][method_name][
                sim_id
            ]
            true_props = results_exp3_true_props[prop_noise][sim_id]
            proportion_nrmse = nrmse(true_props, estimated_props)
            results_exp3_proportions[prop_noise][method_name].append(proportion_nrmse)

In [ ]:
# Plot the NRMSE of P(sig | ctype) for Experiment 3
fig, axes = plt.subplots(
    5,
    len(prop_estimate_noise_grid),
    figsize=(4 * len(prop_estimate_noise_grid), 4 * 5),
    sharey="row",
)

for row_idx, read_length in enumerate(range(1, 6)):
    for col_idx, prop_noise in enumerate(prop_estimate_noise_grid):
        ax = axes[row_idx, col_idx]
        data_to_plot = [
            results_exp3_sig_given_ctype[prop_noise][read_length][method]
            for method in methods_exp3
        ]
        bp = ax.boxplot(data_to_plot, patch_artist=True)
        for patch, color in zip(bp["boxes"], colors_exp3):
            patch.set_facecolor(color)
        for median in bp["medians"]:
            median.set_color("black")
        ax.set_ylim(0, 1.0)
        ax.grid(axis="y", linestyle="--", alpha=0.7)
        ax.set_xticks([])
        if row_idx == 0:
            ax.set_title(f"Prop. estimate noise std\n= {prop_noise:.2f}")
        ax.set_ylabel(f"Read length={read_length}\nNRMSE" if col_idx == 0 else "")

legend_handles = [
    Patch(facecolor=color, edgecolor="black", label=label)
    for color, label in zip(colors_exp3, method_labels_exp3)
]
fig.legend(
    handles=legend_handles,
    title="Methods",
    loc="center left",
    bbox_to_anchor=(0.85, 0.5),
    frameon=True,
    fancybox=True,
    framealpha=1.0,
    borderpad=0.8,
)
fig.suptitle(
    f"Varying proportion estimate noise: NRMSE of estimated P(signature | ctype) vs "
    f"proportion estimate noise std, per pattern length and method"
    f"\nSetting: {SETTING}, {N_CTYPES} cell types ({selected_ctypes}), "
    f"fixed noise level: {FIXED_NOISE_LEVEL}, "
    f"#simulations: {N_SIMULATIONS}, "
    f"max iterations: {MAX_NUM_ITERATIONS}, delta tol: {DELTA_TOL}",
    y=1.0,
)
plt.tight_layout(rect=[0, 0, 0.86, 1])
plt.savefig(
    Path(
        f"./output/noisy_samples_varying_prop_noise_nrmse_setting={SETTING}_noisy-samples={N_NOISY_SAMPLES}_"
        f"sample-size={SAMPLE_SIZE}_max-iterations={MAX_NUM_ITERATIONS}_delta-tol={DELTA_TOL}_fixed-noise-level={FIXED_NOISE_LEVEL}.pdf"
    ),
    bbox_inches="tight",
)
plt.show()

In [ ]:
# Plot the NRMSE of the estimated proportions for Experiment 3
fig, axes = plt.subplots(
    1,
    len(prop_estimate_noise_grid),
    figsize=(4 * len(prop_estimate_noise_grid), 4),
    sharey=True,
)
if len(prop_estimate_noise_grid) == 1:
    axes = [axes]

for col_idx, prop_noise in enumerate(prop_estimate_noise_grid):
    ax = axes[col_idx]
    data_to_plot = [
        results_exp3_proportions[prop_noise][method] for method in methods_exp3
    ]
    bp = ax.boxplot(data_to_plot, patch_artist=True)
    for patch, color in zip(bp["boxes"], colors_exp3):
        patch.set_facecolor(color)
    for median in bp["medians"]:
        median.set_color("black")
    ax.set_ylim(0, 1.0)
    ax.grid(axis="y", linestyle="--", alpha=0.7)
    ax.set_xticks([])
    ax.set_title(f"Prop. estimate noise std\n= {prop_noise:.2f}")
    if col_idx == 0:
        ax.set_ylabel("NRMSE of estimated proportions")

legend_handles = [
    Patch(facecolor=color, edgecolor="black", label=label)
    for color, label in zip(colors_exp3, method_labels_exp3)
]
fig.legend(
    handles=legend_handles,
    title="Methods",
    loc="center left",
    bbox_to_anchor=(0.9, 0.5),
    frameon=True,
    fancybox=True,
    framealpha=1.0,
    borderpad=0.8,
)
fig.suptitle(
    "Exp 3 (varying proportion estimate noise): NRMSE of the estimated ctype proportions vs "
    "proportion estimate noise std, per method"
    f"\nSetting: {SETTING}, fixed noise level: {FIXED_NOISE_LEVEL}, "
    f"#simulations: {N_SIMULATIONS}, "
    f"max iterations: {MAX_NUM_ITERATIONS}, delta tol: {DELTA_TOL}",
    y=1.02,
)
plt.savefig(
    Path(
        f"./output/noisy_samples_varying_prop_noise_ctype_prop_nrmse_setting={SETTING}_noisy-samples={N_NOISY_SAMPLES}_"
        f"sample-size={SAMPLE_SIZE}_max-iterations={MAX_NUM_ITERATIONS}_delta-tol={DELTA_TOL}_fixed-noise-level={FIXED_NOISE_LEVEL}.pdf"
    ),
    bbox_inches="tight",
)
plt.tight_layout(rect=[0, 0, 0.88, 1])
plt.show()

In [ ]:
# Plot execution time ratios (generalized/archetype) for Experiment 3
fig, ax = plt.subplots(figsize=(8, 6))
data_to_plot = [
    time_ratios_exp3_per_prop_noise[prop_noise]
    for prop_noise in prop_estimate_noise_grid
]
bp = ax.boxplot(data_to_plot, patch_artist=True)
for patch, color in zip(bp["boxes"], ["tab:blue"] * len(prop_estimate_noise_grid)):
    patch.set_facecolor(color)
for median in bp["medians"]:
    median.set_color("black")
ax.set_xticks(range(1, len(prop_estimate_noise_grid) + 1))
ax.set_xticklabels([f"{prop_noise:.2f}" for prop_noise in prop_estimate_noise_grid])
ax.set_xlabel("Proportion estimate noise std")
ax.set_ylabel("Execution time ratio (generalized/archetype)")
ax.set_title(
    "Exp 3: execution time ratio (generalized/archetype) vs proportion estimate noise std"
    f"\nSetting: {SETTING}, {N_CTYPES} cell types ({selected_ctypes}), "
    f"#noisy samples: {N_NOISY_SAMPLES}, sample size: {SAMPLE_SIZE}, "
    f"\n#simulations: {N_SIMULATIONS}, fixed noise level: {FIXED_NOISE_LEVEL}, "
    f"max iterations: {MAX_NUM_ITERATIONS}, delta tol: {DELTA_TOL}"
)
ax.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.savefig(
    Path(
        f"./output/noisy_samples_varying_prop_noise_time_ratio_setting={SETTING}_"
        f"noisy-samples={N_NOISY_SAMPLES}_sample-size={SAMPLE_SIZE}_"
        f"max-iterations={MAX_NUM_ITERATIONS}_delta-tol={DELTA_TOL}_"
        f"fixed-noise-level={FIXED_NOISE_LEVEL}.pdf"
    ),
    bbox_inches="tight",
)
plt.show()

## Experiment 4: execution time of the methods

In [ ]:
# Experiment 4 configuration: benchmark #iterations and runtime vs delta_tol.
# All parameters needed to run Experiment 4 independently are defined here.
EXP4_NOISE_LEVEL = (
    0.3  # fixed contamination noise of the ctype proportions in the samples
)
EXP4_PROP_ESTIMATE_NOISE_STD = (
    0.3  # fixed noise on the proportion estimate given to the methods
)
EXP4_N_SIMULATIONS = 30
EXP4_N_NOISY_SAMPLES = 10
EXP4_SAMPLE_SIZE = 200
EXP4_STD_NOISE_ON_NOISE = 0.1  # std of the noise added to the noise level
# high enough that it is never reached (so convergence is driven by delta_tol)
EXP4_MAX_NUM_ITERATIONS = 100000
# grid of convergence thresholds to benchmark
exp4_delta_tol_grid = np.array([1e-2, 1e-3, 1e-4, 1e-5, 1e-6])

methods_exp4 = ["generalized_naive_sl_without_pooling", "sl_with_simple_archetypes"]
method_labels_exp4 = ["Generalized naive\n(no pooling)", "Simple\narchetypes"]
colors_exp4 = ["tab:blue", "tab:red"]

In [ ]:
rng = np.random.default_rng(0)

# results[delta_tol][method] -> list of values across simulations
results_exp4_num_iterations = {
    delta_tol: {method: [] for method in methods_exp4}
    for delta_tol in exp4_delta_tol_grid
}
results_exp4_total_time = {
    delta_tol: {method: [] for method in methods_exp4}
    for delta_tol in exp4_delta_tol_grid
}
results_exp4_time_per_iteration = {
    delta_tol: {method: [] for method in methods_exp4}
    for delta_tol in exp4_delta_tol_grid
}

n_loops = len(exp4_delta_tol_grid) * EXP4_N_SIMULATIONS
with tqdm(total=n_loops, desc="Exp 4: #iterations and runtime vs delta_tol") as pbar:
    for delta_tol in exp4_delta_tol_grid:
        for sim_id in range(EXP4_N_SIMULATIONS):
            ctype_prop_per_sample = build_ctype_prop_per_sample(
                EXP4_NOISE_LEVEL,
                EXP4_STD_NOISE_ON_NOISE,
                EXP4_N_NOISY_SAMPLES,
                N_CTYPES,
            )
            sample_sizes = np.full(EXP4_N_NOISY_SAMPLES, EXP4_SAMPLE_SIZE, dtype=int)
            counts_sig_per_sample = generate_noisy_samples_counts(
                og_dataset,
                selected_ctypes,
                ctype_prop_per_sample,
                sample_sizes,
                signature_positions,
                N_CTYPES,
            )

            # noisy estimate of the proportions given to the methods
            noisy_prop_estimate = exp3_perturb_proportions(
                ctype_prop_per_sample, EXP4_PROP_ESTIMATE_NOISE_STD, rng
            )

            for method_name in methods_exp4:
                start_method_time = time()
                if method_name == "generalized_naive_sl_without_pooling":
                    _, _, num_iterations = generalized_naive_sl_without_pooling(
                        counts_sig_per_sample,
                        noisy_prop_estimate,
                        reestimate_ctype_proba_per_sample=True,
                        max_num_iterations=EXP4_MAX_NUM_ITERATIONS,
                        delta_tol=delta_tol,
                        return_num_iterations=True,
                    )
                elif method_name == "sl_with_simple_archetypes":
                    _, _, num_iterations = sl_with_simple_archetypes(
                        counts_sig_per_sample,
                        noisy_prop_estimate,
                        reestimate_ctype_proba_per_sample=True,
                        max_num_iterations=EXP4_MAX_NUM_ITERATIONS,
                        delta_tol=delta_tol,
                        return_num_iterations=True,
                    )
                else:
                    raise ValueError(f"Unknown method: {method_name}")
                elapsed_time = time() - start_method_time

                results_exp4_num_iterations[delta_tol][method_name].append(
                    num_iterations
                )
                results_exp4_total_time[delta_tol][method_name].append(elapsed_time)
                results_exp4_time_per_iteration[delta_tol][method_name].append(
                    elapsed_time / max(num_iterations, 1)
                )
            pbar.update(1)

In [ ]:
# Plot #iterations, time per iteration and total time vs delta_tol (Experiment 4)
metrics_exp4 = [
    ("Number of iterations", results_exp4_num_iterations),
    ("Time per iteration (s)", results_exp4_time_per_iteration),
    ("Total time (s)", results_exp4_total_time),
]

fig, axes = plt.subplots(3, 1, figsize=(2 * len(exp4_delta_tol_grid) + 4, 12))

n_methods = len(methods_exp4)
group_width = 0.8
box_width = group_width / n_methods
positions_base = np.arange(len(exp4_delta_tol_grid))

use_log_scale_where_appropriate = False
for row_idx, (ylabel, results) in enumerate(metrics_exp4):
    ax = axes[row_idx]
    for m_idx, method in enumerate(methods_exp4):
        data_to_plot = [results[delta_tol][method] for delta_tol in exp4_delta_tol_grid]
        offset = (m_idx - (n_methods - 1) / 2) * box_width
        positions = positions_base + offset
        bp = ax.boxplot(
            data_to_plot,
            positions=positions,
            widths=box_width * 0.9,
            patch_artist=True,
            manage_ticks=False,
        )

        for patch in bp["boxes"]:
            patch.set_facecolor(colors_exp4[m_idx])
        for median in bp["medians"]:
            median.set_color("black")
    if use_log_scale_where_appropriate:
        if row_idx == 0 or row_idx == 2:
            ax.set_yscale("log")
    ax.set_xticks(positions_base)
    ax.set_xticklabels([f"{delta_tol:.0e}" for delta_tol in exp4_delta_tol_grid])
    ax.set_xlabel("delta_tol")
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", linestyle="--", alpha=0.7)

legend_handles = [
    Patch(facecolor=color, edgecolor="black", label=label)
    for color, label in zip(colors_exp4, method_labels_exp4)
]
fig.legend(
    handles=legend_handles,
    title="Methods",
    loc="upper right",
    frameon=True,
    fancybox=True,
    framealpha=1.0,
    borderpad=0.8,
)
fig.suptitle(
    "Exp 4: #iterations, time per iteration and total time vs delta_tol, per method"
    f"\nSetting: {SETTING}, {N_CTYPES} cell types ({selected_ctypes}), "
    f"noise level: {EXP4_NOISE_LEVEL}, prop. estimate noise std: {EXP4_PROP_ESTIMATE_NOISE_STD}, "
    f"\n#noisy samples: {EXP4_N_NOISY_SAMPLES}, sample size: {EXP4_SAMPLE_SIZE}, "
    f"#simulations: {EXP4_N_SIMULATIONS}, max iterations: {EXP4_MAX_NUM_ITERATIONS}",
    y=1.0,
)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(
    Path(
        f"./output/noisy_samples_exp4_iterations_runtime_vs_delta_tol_setting={SETTING}_"
        f"noisy-samples={EXP4_N_NOISY_SAMPLES}_sample-size={EXP4_SAMPLE_SIZE}_"
        f"simulations={EXP4_N_SIMULATIONS}_noise-level={EXP4_NOISE_LEVEL}_"
        f"prop-estimate-noise-std={EXP4_PROP_ESTIMATE_NOISE_STD}_with-log-scale={use_log_scale_where_appropriate}.pdf"
    ),
    bbox_inches="tight",
)
plt.show()

## Experiment 5